In [0]:
# Load the taxi dataset
df = spark.table("yellow_trip_data.yellow.yellow_tripdata_2024_01")

In [0]:
# ============================================================
# 01_delta_lake_internals.py
# Phase 1 — Delta Lake Storage Format Deep Dive
# ============================================================

# Write the taxi data as a Delta table
df.write.format("delta").mode("overwrite").saveAsTable("yellow_trip_data.yellow.taxi_delta")


In [0]:

# -----------------------------------------------------------
# STEP 1: Query the Delta table metadata
# -----------------------------------------------------------
# For Unity Catalog managed tables, inspect using Delta Lake APIs
display(spark.sql("DESCRIBE DETAIL yellow_trip_data.yellow.taxi_delta"))
# You'll see: format=delta, numFiles, sizeInBytes, and other Delta metadata

# View the transaction log history
display(spark.sql("DESCRIBE HISTORY yellow_trip_data.yellow.taxi_delta"))
# You'll see: version 0 (initial write), timestamp, operation details


In [0]:

# -----------------------------------------------------------
# STEP 2: Query the Delta transaction log
# -----------------------------------------------------------
# For Unity Catalog managed tables, access log details via SQL
log_details = spark.sql("""SELECT * FROM (DESCRIBE HISTORY yellow_trip_data.yellow.taxi_delta) WHERE version = 0""")
display(log_details)
# Look at operationMetrics: numOutputRows, numOutputBytes, numFiles
# This shows what was written in the initial transaction


In [0]:

# -----------------------------------------------------------
# STEP 3: Make a change -> creates a NEW log entry (versioning)
# -----------------------------------------------------------
spark.sql("""
    UPDATE yellow_trip_data.yellow.taxi_delta
    SET fare_amount = fare_amount * 1.0
    WHERE fare_amount < 0
""")
# Check the history again -> version 1 appears

display(spark.sql("DESCRIBE HISTORY yellow_trip_data.yellow.taxi_delta"))


In [0]:

# -----------------------------------------------------------
# STEP 4: Time Travel — query an older version
# -----------------------------------------------------------
spark.sql("DESCRIBE HISTORY yellow_trip_data.yellow.taxi_delta").show(truncate=False)

old_version = spark.read.format("delta").option("versionAsOf", 0).table("yellow_trip_data.yellow.taxi_delta")
print(f"Version 0 row count: {old_version.count()}")

new_version = spark.read.format("delta").table("yellow_trip_data.yellow.taxi_delta")
print(f"Latest version row count: {new_version.count()}")


In [0]:

# -----------------------------------------------------------
# STEP 5: Predicate Pushdown demonstration
# -----------------------------------------------------------
# Run this and check Spark UI -> notice "files pruned" in the scan node
high_fares = spark.read.format("delta").table("yellow_trip_data.yellow.taxi_delta") \
    .filter("fare_amount > 200")
high_fares.explain(True)
# In the physical plan, look for "PartitionFilters" / file pruning stats
# This proves the engine skipped files based on _delta_log min/max stats


In [0]:

# -----------------------------------------------------------
# STEP 6: OPTIMIZE and VACUUM
# -----------------------------------------------------------
spark.sql("OPTIMIZE yellow_trip_data.yellow.taxi_delta")
# Compacts small files into larger ones for better query performance

# VACUUM removes old files no longer referenced (minimum 168 hours retention)
spark.sql("VACUUM yellow_trip_data.yellow.taxi_delta")